# Progress Review 2 — Spark Reads Raw Kafka Data from MinIO

This notebook reads the Kafka-streamed weather events written during Progress Review 1.

Flow:

MinIO `raw/kafka_weather_events/` → Python MinIO client → Spark DataFrame

This proves that Spark can process data stored in the MinIO lakehouse environment.

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")
        existing_sc.stop()
        print("Stopped existing SparkContext.")
    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")
    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
import json
import boto3
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("aviation-pr2-read-minio-raw")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)

Spark version: 4.0.0
Spark app ID: app-20260425121024-0022


In [4]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
RAW_BUCKET = os.getenv("RAW_BUCKET", "raw")

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=AWS_REGION,
)

print("MinIO endpoint:", MINIO_ENDPOINT)
print("Raw bucket:", RAW_BUCKET)
print("Available buckets:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

MinIO endpoint: http://minio:9000
Raw bucket: raw
Available buckets: ['lakehouse', 'mlflow', 'raw', 'warehouse']


In [5]:
def list_objects(bucket: str, prefix: str):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])

    return keys


def read_jsonl_objects(bucket: str, keys):
    rows = []

    for key in keys:
        body = (
            s3.get_object(Bucket=bucket, Key=key)["Body"]
            .read()
            .decode("utf-8")
        )

        for line in body.strip().splitlines():
            if not line.strip():
                continue

            row = json.loads(line)

            # Keep only real weather events, not accidental test messages.
            if "event_id" not in row or "airport" not in row or "timestamp_utc" not in row:
                continue

            row["_source_object"] = key
            rows.append(row)

    return rows

In [6]:
raw_keys = list_objects(RAW_BUCKET, "kafka_weather_events/")

print("Raw streamed object count:", len(raw_keys))

for key in raw_keys[-10:]:
    print(key)

if not raw_keys:
    raise RuntimeError("No kafka_weather_events found in MinIO. Run PR1 producer/consumer first.")

Raw streamed object count: 8
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00001.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00002.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00003.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00004.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00005.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00006.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00007.jsonl
kafka_weather_events/run_id=20260425T094031Z-d767cd57/year=2026/month=04/day=25/part-00008.jsonl


In [7]:
rows = read_jsonl_objects(RAW_BUCKET, raw_keys)

pdf = pd.DataFrame(rows)

print("Valid weather rows loaded:", len(pdf))
display(pdf.head())

if pdf.empty:
    raise RuntimeError("No valid weather rows found after filtering.")

Valid weather rows loaded: 72


,event_id,airport,timestamp_utc,latitude,longitude,temperature_k,temperature_c,wind_u_ms,wind_v_ms,wind_speed_ms,wind_speed_kts,precipitation_m,precipitation_mm,surface_pressure_pa,cape_j_kg,_kafka_topic,_kafka_partition,_kafka_offset,_ingested_at_utc,_source_object
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,40.75,286.25,282.574158,9.424164,-0.005779,1.871856,1.871865,3.638606,5.761161e-06,0.005761,101271.765625,12.700439,weather.raw,0,0,2026-04-25T09:40:41.373487+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,40.75,286.25,282.615082,9.465088,0.255980,2.030057,2.046132,3.977354,1.056492e-05,0.010565,101288.007812,21.743164,weather.raw,0,1,2026-04-25T09:40:41.373644+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,40.75,286.25,282.552124,9.402130,-0.076426,2.341874,2.343121,4.554652,1.248531e-05,0.012485,101208.484375,23.470459,weather.raw,0,2,2026-04-25T09:40:41.373665+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,40.75,286.25,282.542694,9.392700,0.014148,2.136289,2.136336,4.152695,1.439825e-06,0.001440,101199.937500,17.780762,weather.raw,0,3,2026-04-25T09:40:41.373677+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,40.75,286.25,282.262512,9.112518,0.130988,2.419828,2.423371,4.710645,9.592623e-07,0.000959,101201.640625,14.834229,weather.raw,0,4,2026-04-25T09:40:41.373688+00:00,kafka_weather_events/run_id=20260425T094031Z-d...


In [8]:
raw_spark_df = spark.createDataFrame(pdf)

raw_spark_df.printSchema()
raw_spark_df.show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- airport: string (nullable = true)
 |-- timestamp_utc: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature_k: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- wind_u_ms: double (nullable = true)
 |-- wind_v_ms: double (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- wind_speed_kts: double (nullable = true)
 |-- precipitation_m: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- surface_pressure_pa: double (nullable = true)
 |-- cape_j_kg: double (nullable = true)
 |-- _kafka_topic: string (nullable = true)
 |-- _kafka_partition: long (nullable = true)
 |-- _kafka_offset: long (nullable = true)
 |-- _ingested_at_utc: string (nullable = true)
 |-- _source_object: string (nullable = true)

+-----------------------+-------+-------------------+--------+---------+------------------+-----------------+---

In [9]:
raw_spark_df.groupBy("airport").count().show()

raw_spark_df.select(
    F.min("timestamp_utc").alias("min_timestamp"),
    F.max("timestamp_utc").alias("max_timestamp"),
    F.count("*").alias("row_count"),
).show(truncate=False)

+-------+-----+
|airport|count|
+-------+-----+
|    JFK|   72|
+-------+-----+

+-------------------+-------------------+---------+
|min_timestamp      |max_timestamp      |row_count|
+-------------------+-------------------+---------+
|2022-01-01 00:00:00|2022-01-03 23:00:00|72       |
+-------------------+-------------------+---------+



In [10]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
